# Phase 3 — Duplicate Investigation & Leakage-Safe Evaluation

Determine whether duplicate structure in the phishing-website benchmark creates train/test leakage and how much it affects the current ML evaluation.

**Experiment only:** this notebook does not modify production pipeline code, remove rows, change preprocessing, or tune the model. It compares the existing random row split with a duplicate-group-aware holdout using the seeded Random Forest established in Phase 2.

### Dataset under test
- Dataset: `data/raw/phisingData.csv`
- Target: `Result` (`-1`, `1`)
- Feature columns: all columns except `Result`
- Existing split: `test_size=0.2`, `random_state=42`, no stratification
- Controlled model: `RandomForestClassifier(n_estimators=128, criterion='gini', bootstrap=True, max_depth=None, max_features='sqrt', random_state=42)`

### Core questions
1. How many exact duplicate rows and duplicate feature groups exist?
2. Are duplicate feature vectors label-consistent?
3. How large are duplicate groups?
4. How much feature-identical overlap exists between the current train and test sets?
5. What is model performance on the current random row split?
6. What changes when duplicate groups are kept entirely within train or test?
7. What should be changed in the production pipeline, if anything?

## Methodological guardrails

This experiment intentionally changes **only the split strategy**. Both evaluations use the same raw dataset, KNN imputation configuration, Random Forest hyperparameters, and model seed.

We will not infer that duplicates should be deleted merely because they exist. Duplicate rows may represent legitimate repeated observations in a benchmark dataset. The experiment first establishes their structure and their effect on evaluation.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split

# Resolve paths so the notebook works when opened from the repository root or notebooks/.
ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
DATASET_RELATIVE = Path("data") / "raw" / "phisingData.csv"
for candidate in ROOT_CANDIDATES:
    if (candidate / DATASET_RELATIVE).exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not locate data/raw/phisingData.csv from the notebook working directory. "
        "Run the notebook from the repository root or notebooks/ directory."
    )

DATASET_PATH = ROOT / DATASET_RELATIVE
EVALUATION_DIR = ROOT / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = EVALUATION_DIR / "duplicate_leakage_baseline.json"

TARGET = "Result"
TEST_SIZE = 0.20
RANDOM_STATE = 42
N_NEIGHBORS = 3
IMPUTER_WEIGHTS = "uniform"

RF_PARAMS = {
    "n_estimators": 128,
    "criterion": "gini",
    "bootstrap": True,
    "max_depth": None,
    "max_features": "sqrt",
    "random_state": RANDOM_STATE,
}

print("Repository root:", ROOT)
print("Dataset:", DATASET_PATH)
print("Output:", OUTPUT_PATH)

Repository root: E:\Projects\Network security log triage agent\notebooks
Dataset: E:\Projects\Network security log triage agent\notebooks\data\raw\phisingData.csv
Output: E:\Projects\Network security log triage agent\notebooks\evaluation\duplicate_leakage_baseline.json


## A. Load and fingerprint the dataset

In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def dataframe_fingerprint(frame: pd.DataFrame) -> str:
    payload = pd.util.hash_pandas_object(frame, index=True).to_numpy().tobytes()
    columns = "|".join(map(str, frame.columns)).encode("utf-8")
    return hashlib.sha256(columns + payload).hexdigest()


def feature_group_keys(frame: pd.DataFrame, feature_columns: list[str]) -> pd.Series:
    # Hash only feature values; Result is deliberately excluded so conflicting labels
    # for identical feature vectors can be detected.
    hashed = pd.util.hash_pandas_object(frame[feature_columns], index=False)
    return hashed.astype("uint64").astype(str)


df = pd.read_csv(DATASET_PATH)
feature_columns = [column for column in df.columns if column != TARGET]

assert TARGET in df.columns
assert len(feature_columns) == 30

dataset_sha256 = sha256_file(DATASET_PATH)
dataset_fp = dataframe_fingerprint(df)

print("Shape:", df.shape)
print("Target counts:", df[TARGET].value_counts().to_dict())
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Unique full rows:", int(len(df.drop_duplicates())))
print("SHA256:", dataset_sha256)
print("DataFrame fingerprint:", dataset_fp)

Shape: (11055, 31)
Target counts: {1: 6157, -1: 4898}
Exact duplicate rows: 5206
Unique full rows: 5849
SHA256: a4b16abbd8610e4f53fd63e8eb3da793157a961111ed5e9d9d8afa21af866995
DataFrame fingerprint: 44da7c443a4861a2910cadbc57b3a1faea0581567078ab1f6bce0d8ffc09b5ca


## B. Exact duplicate and feature-vector structure

Two different notions are measured:

- **Exact row duplicate:** all 31 columns, including `Result`, are identical.
- **Feature-vector duplicate:** the 30 feature columns are identical, regardless of `Result`.

The second definition is the relevant one for leakage analysis because identical features can appear in both train and test even when the target column is excluded.

In [3]:
feature_keys = feature_group_keys(df, feature_columns)
feature_group_sizes = feature_keys.value_counts(sort=True)

duplicate_feature_rows = int((feature_group_sizes - 1).clip(lower=0).sum())
feature_duplicate_groups = int((feature_group_sizes > 1).sum())
max_feature_group_size = int(feature_group_sizes.max())

label_counts_by_group = df.assign(_feature_group=feature_keys).groupby("_feature_group")[TARGET].nunique()
conflicting_label_groups = int((label_counts_by_group > 1).sum())
consistent_duplicate_groups = int(((feature_group_sizes > 1) & (label_counts_by_group == 1)).sum())

print("Feature-vector duplicate rows (excluding first occurrence per group):", duplicate_feature_rows)
print("Feature-vector duplicate groups:", feature_duplicate_groups)
print("Maximum feature-group size:", max_feature_group_size)
print("Conflicting-label feature groups:", conflicting_label_groups)
print("Label-consistent duplicate feature groups:", consistent_duplicate_groups)

print("\nTop duplicate-group sizes:")
print(feature_group_sizes.head(20).to_string())

print("\nFeature-group size distribution:")
print(feature_group_sizes.value_counts().sort_index().to_string())

Feature-vector duplicate rows (excluding first occurrence per group): 5270
Feature-vector duplicate groups: 2614
Maximum feature-group size: 25
Conflicting-label feature groups: 64
Label-consistent duplicate feature groups: 2550

Top duplicate-group sizes:
4923262558827319503     25
11405546634770831513    24
15468007062444543709    24
8231871721867961930     22
1857076789035290530     16
7879107533947650017     16
2202209043387335316     15
8764330994407845666     13
6732885574429115684     13
9164604418614099480     13
16037453076472507455    12
12310070079366216887    12
8791461925482598373     12
604804585264674631      12
9509336301789854289     11
5075962596989240635     11
6291673333645409862     11
18150750104696948311    11
15680704621072221195    11
8623266305877475301     10

Feature-group size distribution:
count
1     3171
2     1487
3      423
4      437
5       89
6       63
7       30
8       32
9       20
10      14
11       5
12       4
13       3
15       1
16       

## C. Inspect duplicate groups and label consistency

If `conflicting_label_groups == 0`, every feature-identical group has one observed target label. If it is non-zero, those groups require separate data-quality investigation before treating duplicate-aware evaluation as definitive.

In [4]:
group_summary = (
    df.assign(_feature_group=feature_keys)
    .groupby("_feature_group")
    .agg(
        group_size=(TARGET, "size"),
        unique_labels=(TARGET, "nunique"),
        labels=(TARGET, lambda values: tuple(sorted(pd.unique(values)))),
    )
    .sort_values(["group_size", "_feature_group"], ascending=[False, True])
)

group_summary["label_consistent"] = group_summary["unique_labels"] == 1

display(group_summary.head(20))

if conflicting_label_groups:
    print("\nConflicting groups:")
    display(group_summary[~group_summary["label_consistent"]].head(50))
else:
    print("\nNo feature-identical groups with conflicting Result labels were found.")

,group_size,unique_labels,labels,label_consistent
_feature_group,,,,
4923262558827319503,25,1,"(1,)",True
11405546634770831513,24,1,"(1,)",True
15468007062444543709,24,1,"(1,)",True
8231871721867961930,22,1,"(1,)",True
1857076789035290530,16,1,"(-1,)",True
7879107533947650017,16,1,"(1,)",True
2202209043387335316,15,1,"(1,)",True
6732885574429115684,13,1,"(-1,)",True
8764330994407845666,13,2,"(-1, 1)",False



Conflicting groups:


,group_size,unique_labels,labels,label_consistent
_feature_group,,,,
8764330994407845666,13,2,"(-1, 1)",False
604804585264674631,12,2,"(-1, 1)",False
6291673333645409862,11,2,"(-1, 1)",False
11017544998918605160,10,2,"(-1, 1)",False
1743871441200784577,10,2,"(-1, 1)",False
8623266305877475301,10,2,"(-1, 1)",False
1171440625767526928,9,2,"(-1, 1)",False
13296503481488565505,9,2,"(-1, 1)",False
2551413634984213238,9,2,"(-1, 1)",False


## D. Reproduce the current random row split and measure leakage

This is the evaluation design used by the current pipeline: rows are split independently, so duplicate feature vectors are allowed to land in both partitions.

In [5]:
train_df_random, test_df_random = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=None,
)

train_keys_random = set(feature_group_keys(train_df_random, feature_columns))
test_keys_random = feature_group_keys(test_df_random, feature_columns)

test_overlap_mask = test_keys_random.isin(train_keys_random)
train_overlap_mask = feature_group_keys(train_df_random, feature_columns).isin(set(test_keys_random))

test_rows_with_train_duplicate = int(test_overlap_mask.sum())
train_rows_with_test_duplicate = int(train_overlap_mask.sum())
shared_feature_groups = len(train_keys_random.intersection(set(test_keys_random)))

print("Random split shapes:", train_df_random.shape, test_df_random.shape)
print("Shared feature groups:", shared_feature_groups)
print(
    "Test rows with an identical feature vector in train:",
    test_rows_with_train_duplicate,
    f"({test_rows_with_train_duplicate / len(test_df_random):.2%})",
)
print(
    "Train rows with an identical feature vector in test:",
    train_rows_with_test_duplicate,
    f"({train_rows_with_test_duplicate / len(train_df_random):.2%})",
)

Random split shapes: (8844, 31) (2211, 31)
Shared feature groups: 1143
Test rows with an identical feature vector in train: 1447 (65.45%)
Train rows with an identical feature vector in test: 2703 (30.56%)


## E. Evaluation helpers

The preprocessing matches the Phase 2 baseline: fit `KNNImputer(n_neighbors=3, weights='uniform')` on training features only, then transform both partitions. The target is mapped from `{-1, 1}` to `{0, 1}` only for model evaluation.

In [6]:
def prepare_features(
    train_frame: pd.DataFrame, test_frame: pd.DataFrame
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    x_train = train_frame[feature_columns].copy()
    x_test = test_frame[feature_columns].copy()
    y_train = train_frame[TARGET].map({-1: 0, 1: 1}).to_numpy()
    y_test = test_frame[TARGET].map({-1: 0, 1: 1}).to_numpy()

    if pd.isna(y_train).any() or pd.isna(y_test).any():
        raise ValueError("Unexpected target value outside {-1, 1}.")

    imputer = KNNImputer(n_neighbors=N_NEIGHBORS, weights=IMPUTER_WEIGHTS)
    x_train_transformed = imputer.fit_transform(x_train)
    x_test_transformed = imputer.transform(x_test)
    return x_train_transformed, x_test_transformed, y_train, y_test


def evaluate_split(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> dict:
    x_train, x_test, y_train, y_test = prepare_features(train_frame, test_frame)
    model = RandomForestClassifier(**RF_PARAMS)
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)

    cm = confusion_matrix(y_test, predictions, labels=[0, 1])
    return {
        "train_shape": list(train_frame.shape),
        "test_shape": list(test_frame.shape),
        "train_target_distribution": {str(k): int(v) for k, v in train_frame[TARGET].value_counts().sort_index().items()},
        "test_target_distribution": {str(k): int(v) for k, v in test_frame[TARGET].value_counts().sort_index().items()},
        "accuracy": float(accuracy_score(y_test, predictions)),
        "f1": float(f1_score(y_test, predictions)),
        "precision": float(precision_score(y_test, predictions)),
        "recall": float(recall_score(y_test, predictions)),
        "confusion_matrix": cm.tolist(),
    }

## F. Baseline model on the current random split

In [7]:
random_metrics = evaluate_split(train_df_random, test_df_random)
print(json.dumps(random_metrics, indent=2))

{
  "train_shape": [
    8844,
    31
  ],
  "test_shape": [
    2211,
    31
  ],
  "train_target_distribution": {
    "-1": 3942,
    "1": 4902
  },
  "test_target_distribution": {
    "-1": 956,
    "1": 1255
  },
  "accuracy": 0.968340117593849,
  "f1": 0.9723320158102767,
  "precision": 0.9647058823529412,
  "recall": 0.9800796812749004,
  "confusion_matrix": [
    [
      911,
      45
    ],
    [
      25,
      1230
    ]
  ]
}


## G. Duplicate-group-aware split

`GroupShuffleSplit` assigns an entire feature-duplicate group to one partition. Therefore, identical feature vectors cannot cross the train/test boundary.

The requested test fraction is approximately 20%; because groups have different sizes, the resulting test row count may not be exactly 20%. That is expected for group-based holdout evaluation.

In [8]:
groups = feature_keys
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, df[TARGET], groups=groups))

train_df_group = df.iloc[train_idx].copy()
test_df_group = df.iloc[test_idx].copy()

train_group_keys = set(feature_group_keys(train_df_group, feature_columns))
test_group_keys = set(feature_group_keys(test_df_group, feature_columns))
shared_group_keys = train_group_keys.intersection(test_group_keys)

assert len(shared_group_keys) == 0, "Group-aware split leaked duplicate feature groups across partitions."

print("Group-aware split shapes:", train_df_group.shape, test_df_group.shape)
print("Train fraction:", len(train_df_group) / len(df))
print("Test fraction:", len(test_df_group) / len(df))
print("Shared feature groups:", len(shared_group_keys))
print("Train target distribution:", train_df_group[TARGET].value_counts().sort_index().to_dict())
print("Test target distribution:", test_df_group[TARGET].value_counts().sort_index().to_dict())

Group-aware split shapes: (8751, 31) (2304, 31)
Train fraction: 0.7915875169606513
Test fraction: 0.2084124830393487
Shared feature groups: 0
Train target distribution: {-1: 3949, 1: 4802}
Test target distribution: {-1: 949, 1: 1355}


## H. Leakage-safe model evaluation

In [9]:
group_metrics = evaluate_split(train_df_group, test_df_group)
print(json.dumps(group_metrics, indent=2))

{
  "train_shape": [
    8751,
    31
  ],
  "test_shape": [
    2304,
    31
  ],
  "train_target_distribution": {
    "-1": 3949,
    "1": 4802
  },
  "test_target_distribution": {
    "-1": 949,
    "1": 1355
  },
  "accuracy": 0.9453125,
  "f1": 0.9527736131934033,
  "precision": 0.968012185833968,
  "recall": 0.9380073800738007,
  "confusion_matrix": [
    [
      907,
      42
    ],
    [
      84,
      1271
    ]
  ]
}


## I. Controlled comparison

The comparison below changes the split strategy while holding preprocessing and model configuration constant. It is descriptive evidence, not a model-optimization result.

In [10]:
comparison = pd.DataFrame(
    [
        {
            "evaluation": "Random row split",
            "test_rows": len(test_df_random),
            "shared_feature_groups": shared_feature_groups,
            "test_duplicate_overlap_rows": test_rows_with_train_duplicate,
            **{metric: random_metrics[metric] for metric in ["accuracy", "precision", "recall", "f1"]},
        },
        {
            "evaluation": "Duplicate-group split",
            "test_rows": len(test_df_group),
            "shared_feature_groups": len(shared_group_keys),
            "test_duplicate_overlap_rows": 0,
            **{metric: group_metrics[metric] for metric in ["accuracy", "precision", "recall", "f1"]},
        },
    ]
)

display(comparison)

print("\nF1 difference (group-aware minus random):", group_metrics["f1"] - random_metrics["f1"])
print("Recall difference:", group_metrics["recall"] - random_metrics["recall"])
print("Precision difference:", group_metrics["precision"] - random_metrics["precision"])

,evaluation,test_rows,shared_feature_groups,test_duplicate_overlap_rows,accuracy,precision,recall,f1
0,Random row split,2211,1143,1447,0.968340,0.964706,0.980080,0.972332
1,Duplicate-group split,2304,0,0,0.945312,0.968012,0.938007,0.952774



F1 difference (group-aware minus random): -0.01955840261687347
Recall difference: -0.04207230120109973
Precision difference: 0.0033063034810267844


## J. Optional diagnostic: duplicate overlap by target

This does not alter evaluation. It helps show whether the feature-identical test rows crossing into training are concentrated in one target class.

In [11]:
random_test_with_overlap = test_df_random.copy()
random_test_with_overlap["_feature_group"] = feature_group_keys(random_test_with_overlap, feature_columns)
random_test_with_overlap["_has_train_duplicate"] = random_test_with_overlap["_feature_group"].isin(train_keys_random)

overlap_by_target = (
    random_test_with_overlap.groupby(TARGET)["_has_train_duplicate"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "overlap_rows", "count": "test_rows"})
)
overlap_by_target["overlap_rate"] = overlap_by_target["overlap_rows"] / overlap_by_target["test_rows"]
display(overlap_by_target)

,overlap_rows,test_rows,overlap_rate
Result,,,
-1,528,956,0.552301
1,919,1255,0.732271


## K. Persist the experiment result

The JSON artifact captures the dataset identity, duplicate structure, leakage measurements, split configurations, model configuration, and both evaluation results so the Phase 3 conclusion can be reviewed without rerunning the notebook.

In [12]:
environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
}

result = {
    "experiment": {
        "name": "phase_3_duplicate_leakage_baseline",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": "completed",
        "production_code_modified": False,
    },
    "dataset": {
        "path": str(DATASET_PATH),
        "sha256": dataset_sha256,
        "shape": list(df.shape),
        "dataframe_fingerprint": dataset_fp,
        "target": TARGET,
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "unique_full_rows": int(len(df.drop_duplicates())),
        "feature_columns": len(feature_columns),
    },
    "duplicate_structure": {
        "feature_duplicate_rows": duplicate_feature_rows,
        "feature_duplicate_groups": feature_duplicate_groups,
        "max_feature_group_size": max_feature_group_size,
        "conflicting_label_groups": conflicting_label_groups,
        "consistent_duplicate_groups": consistent_duplicate_groups,
        "group_size_distribution": {str(k): int(v) for k, v in feature_group_sizes.value_counts().sort_index().items()},
    },
    "random_row_split": {
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "stratified": False,
        "train_shape": list(train_df_random.shape),
        "test_shape": list(test_df_random.shape),
        "shared_feature_groups": shared_feature_groups,
        "test_rows_with_train_duplicate": test_rows_with_train_duplicate,
        "test_duplicate_overlap_rate": test_rows_with_train_duplicate / len(test_df_random),
        "train_rows_with_test_duplicate": train_rows_with_test_duplicate,
        "train_duplicate_overlap_rate": train_rows_with_test_duplicate / len(train_df_random),
        "metrics": random_metrics,
    },
    "duplicate_group_split": {
        "splitter": "GroupShuffleSplit",
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "train_shape": list(train_df_group.shape),
        "test_shape": list(test_df_group.shape),
        "shared_feature_groups": len(shared_group_keys),
        "metrics": group_metrics,
    },
    "model": {
        "algorithm": "RandomForestClassifier",
        "params": RF_PARAMS,
        "preprocessing": {
            "transformer": "KNNImputer",
            "n_neighbors": N_NEIGHBORS,
            "weights": IMPUTER_WEIGHTS,
        },
    },
    "environment": environment,
}

OUTPUT_PATH.write_text(json.dumps(result, indent=2), encoding="utf-8")
print(f"Wrote: {OUTPUT_PATH}")
print(json.dumps(result, indent=2))

Wrote: E:\Projects\Network security log triage agent\notebooks\evaluation\duplicate_leakage_baseline.json
{
  "experiment": {
    "name": "phase_3_duplicate_leakage_baseline",
    "generated_at_utc": "2026-09-21T18:39:27.703259+00:00",
    "status": "completed",
    "production_code_modified": false
  },
  "dataset": {
    "path": "E:\\Projects\\Network security log triage agent\\notebooks\\data\\raw\\phisingData.csv",
    "sha256": "a4b16abbd8610e4f53fd63e8eb3da793157a961111ed5e9d9d8afa21af866995",
    "shape": [
      11055,
      31
    ],
    "dataframe_fingerprint": "44da7c443a4861a2910cadbc57b3a1faea0581567078ab1f6bce0d8ffc09b5ca",
    "target": "Result",
    "exact_duplicate_rows": 5206,
    "unique_full_rows": 5849,
    "feature_columns": 30
  },
  "duplicate_structure": {
    "feature_duplicate_rows": 5270,
    "feature_duplicate_groups": 2614,
    "max_feature_group_size": 25,
    "conflicting_label_groups": 64,
    "consistent_duplicate_groups": 2550,
    "group_size_distrib

## L. Interpretation checklist

After execution, record the conclusion from the measured values rather than assuming that duplicates are inherently invalid.

### Decision rules
- If conflicting-label groups exist: flag a data-quality issue for investigation.
- If random-split test overlap is non-zero: document that the current evaluation permits duplicate leakage.
- If group-aware performance differs materially from the random split: use the comparison to qualify the baseline's generalization claim.
- If group-aware performance is similar: duplicate leakage has less observable effect on this particular holdout, but the group-aware result remains the cleaner estimate for duplicate-structured data.
- Do **not** delete duplicates or alter production splitting solely from this notebook. Any such change belongs in a subsequent controlled experiment.

### Expected Phase 3 output

The next step after reviewing this artifact is to decide whether v0.2.x should introduce duplicate-aware validation, a documented dataset policy, or a separate benchmark protocol.